# Measure Twice - GRPO on top of the SFT LoRA
One-step GRPO: each prompt is an episode prefix from `data/grpo.jsonl`; the reward is `reward.step_reward` on the model's next turn.

**Budget.** No vLLM on T4, so generation is HF `generate` and dominates step time (~20-40 s/step at 8 x 200 tokens; check the first steps).
`MAX_STEPS=400` targets <= 5 h. Checkpoints every 25 steps go to `/kaggle/working/grpo_ckpt`. Running with **Save Version -> Save & Run All**
(background, up to 12 h) is safer than an interactive session.

**Setup.** Kaggle: GPU T4 x2, Internet ON. Attach the repo as a Kaggle Dataset (default mount below) or uncomment the `git clone` line.
The repo is copied to `/kaggle/working/mt` because `/kaggle/input` is read-only (and `gen_sft.py` writes into `data/`).

**SFT adapter:** attach the SFT notebook's output (it contains `sft_lora/`) as an input; the cell below searches the usual places.

In [ ]:
import os, sys, shutil
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Unsloth trains on one GPU; the 2nd T4 stays idle
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO = "/kaggle/input/measure-twice"         # Kaggle Dataset mount of the repo folder
# !git clone https://github.com/<you>/measure-twice /kaggle/working/mt-src && REPO = "/kaggle/working/mt-src"
WORK = "/kaggle/working/mt"
shutil.copytree(REPO, WORK, dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(os.listdir(WORK))

In [ ]:
!pip install -q --upgrade unsloth unsloth_zoo
# if the import below fails after an Unsloth release, pin to the last good versions (pip freeze from a working run)

**Choices.**
- fp16 (T4 has no bf16), `fast_inference=False` (vLLM is unreliable on T4 and would also eat VRAM needed for training).
- Start from SFT: build the *same* LoRA (r=16, same modules) on the 4-bit base, then load the saved SFT adapter weights into it. That leaves a normal trainable Unsloth PEFT model, which is the most reliable path (loading an adapter dir directly can come back frozen).
- `beta=0.0`: with a PEFT model TRL's reference model is the base with the adapter *disabled*, i.e. the pre-SFT model, so a KL term would pull us back toward the untrained base. Not wanted, and skipping the ref forward pass saves time.
- Prompts are pre-rendered to strings with `enable_thinking=False`, exactly like `rollout.hf_generator`, so no reliance on GRPOConfig chat-template kwargs. `history` is stored as a JSON string because Arrow would otherwise merge the varying `args` dicts into one struct and add `None` keys.
- One reward function only: `step_reward` already labels malformed output as `invalid`, so a separate format reward would double-count.

In [ ]:
from unsloth import FastLanguageModel   # import unsloth before trl/transformers so its patches apply
import torch

BASE = "Qwen/Qwen3-1.7B"
MAX_SEQ = 2048
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            use_gradient_checkpointing="unsloth", random_state=0)

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,  # dtype=None -> fp16 on T4 (no bf16)
    fast_inference=False)                                          # no vLLM on T4, see note above
model = FastLanguageModel.get_peft_model(model, **LORA)

In [ ]:
from pathlib import Path
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

cands = ["/kaggle/working/sft_lora", *map(str, Path("/kaggle/input").glob("*/sft_lora")),
         *map(str, Path("/kaggle/input").glob("*/*/sft_lora"))]
SFT_LORA = next(c for c in cands if Path(c, "adapter_model.safetensors").exists())
res = set_peft_model_state_dict(model, load_file(f"{SFT_LORA}/adapter_model.safetensors"))
assert not res.unexpected_keys, res.unexpected_keys[:5]
print("loaded SFT adapter from", SFT_LORA)

In [ ]:
import json, subprocess
from datasets import Dataset

if not Path("data/grpo.jsonl").exists():
    subprocess.run([sys.executable, "gen_sft.py"], check=True)

MAX_PROMPT = 1536
MAX_COMPLETION = 200
rows = []
for l in open("data/grpo.jsonl"):
    r = json.loads(l)
    p = tokenizer.apply_chat_template(r["prompt"], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    if len(tokenizer(p, add_special_tokens=False)["input_ids"]) <= MAX_PROMPT:   # drop, never truncate (would cut the tool list)
        rows.append({"prompt": p, "task_id": r["task_id"], "history": json.dumps(r["history"])})
ds = Dataset.from_list(rows).shuffle(seed=0)
print(len(rows), "prompts")

In [ ]:
import reward
from sandbox import tasks

def reward_fn(completions, task_id, history, **kw):
    out = []
    for c, tid, h in zip(completions, task_id, history):
        text = c if isinstance(c, str) else c[0]["content"]   # str prompts -> str completions; handle chat form too
        out.append(float(reward.step_reward(tasks.get(tid), [tuple(x) for x in json.loads(h)], text)))
    return out

# smoke test on gold-free strings so a contract mismatch fails here, not 10 minutes into training
print(reward_fn(["<tool_call>{\"name\": \"nope\", \"arguments\": {}}</tool_call>", "Done."],
                ds[:2]["task_id"], ds[:2]["history"]))

In [ ]:
import inspect
from trl import GRPOConfig, GRPOTrainer
from unsloth import is_bfloat16_supported

MAX_STEPS = 400
cfg = dict(
    output_dir="/kaggle/working/grpo_ckpt",
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    lr_scheduler_type="cosine", warmup_ratio=0.05,
    weight_decay=0.0, max_grad_norm=0.1,
    optim="adamw_8bit",
    per_device_train_batch_size=8,      # = completions per device step; must be divisible by num_generations
    gradient_accumulation_steps=2,      # -> 2 prompts x 8 completions per optimizer step
    num_generations=8,
    max_prompt_length=MAX_PROMPT,       # dropped automatically on TRL versions that removed it
    max_completion_length=MAX_COMPLETION,
    temperature=1.0,
    beta=0.0,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    logging_steps=1,                    # logs reward, rewards/reward_fn/mean, reward_std, completion length
    save_steps=25, save_total_limit=3,
    report_to="none",
    remove_unused_columns=False,        # reward_fn needs task_id / history
    seed=0,
)
ok = set(inspect.signature(GRPOConfig.__init__).parameters)
print("ignored (not in this TRL):", [k for k in cfg if k not in ok])
args = GRPOConfig(**{k: v for k, v in cfg.items() if k in ok})

FastLanguageModel.for_training(model)
trainer = GRPOTrainer(model=model, processing_class=tokenizer, reward_funcs=[reward_fn], args=args, train_dataset=ds)

**Run / resume.** Fresh run: `RESUME = False`. After a timeout: attach the previous version's output as input, copy its `grpo_ckpt` back
(uncomment the copy line), set `RESUME = True`; the trainer picks the latest `checkpoint-*` (model, optimizer, step count).

In [ ]:
RESUME = False
# shutil.copytree("/kaggle/input/<prev-grpo-output>/grpo_ckpt", "/kaggle/working/grpo_ckpt", dirs_exist_ok=True)
trainer.train(resume_from_checkpoint=True if RESUME else None)

In [ ]:
import pandas as pd
log = pd.DataFrame([h for h in trainer.state.log_history if "reward" in h])
print(log[["step", "reward", "reward_std"]].rolling(10).mean().iloc[::10].to_string())

In [ ]:
GRPO_LORA = "/kaggle/working/grpo_lora"
model.save_pretrained(GRPO_LORA)
tokenizer.save_pretrained(GRPO_LORA)
print(os.listdir(GRPO_LORA))

## Quick val eval (30 tasks) - compare with the SFT numbers

In [ ]:
import statistics as st
import rollout
from sandbox import tasks
from unsloth import FastLanguageModel

def evaluate(model, tokenizer, n=30, tag=""):
    FastLanguageModel.for_inference(model)
    gen = rollout.hf_generator(model, tokenizer, max_new_tokens=256)
    eps = [rollout.run_episode(t, gen) for t in tasks.generate("val", n)]
    keys = ["success", "n_calls", "n_redundant", "n_unsafe", "n_invalid"]
    res = {k: st.mean(float(e[k]) for e in eps) for k in keys}
    print(tag, {k: round(v, 3) for k, v in res.items()})
    return res, eps

In [ ]:
grpo_res, grpo_eps = evaluate(model, tokenizer, 30, "GRPO")

In [ ]:
# Optional: push the adapter to the HF Hub. Add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets); never hardcode tokens.
PUSH = False
HF_REPO = "<your-username>/measure-twice-grpo"
if PUSH:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    model.push_to_hub(HF_REPO, token=tok, private=True)
    tokenizer.push_to_hub(HF_REPO, token=tok, private=True)